Import libraries


In [ ]:
import pandas as pd
import numpy as np
import joblib
import requests
import os
import time
import json
from datetime import datetime, timedelta
from google.colab import files
from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
url="https://raw.githubusercontent.com/Searph-S0708023/Taap_surakshak/refs/heads/main/mumbai_ward_vulnerability_data_with_green_cover_ranked_1_to_5.csv"
vulnerability = pd.read_csv(url)
vulnerability


,ward_name,ward_id,population_2025,household_density_category,population_density_category,socially_weaker_section_category,vulnerability_index_category,marginal_workers_category,illiteracy_rate_category,dilapidated_household_category,...,green_cover_reported_value,green_cover_data_year,green_cover_source,green_cover_source_url,green_cover_data_status,ward_tree_count_2023_24,green_cover_score_1_to_5,green_cover_score_method,ward_tree_count_source,ward_tree_count_source_url
0,A,1,194120,2,2,3,2,4,4,3,...,Not found in cited public summary,WRI/BMC summary based on satellite imagery 201...,WRI India green-cover study shared by BMC/MCAP...,https://indianexpress.com/article/cities/mumba...,Percentage unavailable in accessible summary; ...,83201,2,Proxy only: BMC 2023–24 ward tree count quinti...,"BMC Environment Status Report 2023–24, Table 8.1",https://portal.mcgm.gov.in/irj/go/km/docs/docu...
1,B,2,133616,4,4,2,4,2,4,4,...,<10%,WRI/BMC summary based on satellite imagery 201...,WRI India green-cover study shared by BMC/MCAP...,https://indianexpress.com/article/cities/mumba...,Percentage unavailable in accessible summary; ...,7816,1,Published green-cover band <10%; score 1 (very...,"BMC Environment Status Report 2023–24, Table 8.1",https://portal.mcgm.gov.in/irj/go/km/docs/docu...
2,C,3,174419,5,5,1,5,2,2,3,...,<10%,WRI/BMC summary based on satellite imagery 201...,WRI India green-cover study shared by BMC/MCAP...,https://indianexpress.com/article/cities/mumba...,Percentage unavailable in accessible summary; ...,5756,1,Published green-cover band <10%; score 1 (very...,"BMC Environment Status Report 2023–24, Table 8.1",https://portal.mcgm.gov.in/irj/go/km/docs/docu...
3,D,4,364106,3,3,2,3,4,2,3,...,Not found in cited public summary,WRI/BMC summary based on satellite imagery 201...,WRI India green-cover study shared by BMC/MCAP...,https://indianexpress.com/article/cities/mumba...,Percentage unavailable in accessible summary; ...,100317,3,Proxy only: BMC 2023–24 ward tree count quinti...,"BMC Environment Status Report 2023–24, Table 8.1",https://portal.mcgm.gov.in/irj/go/km/docs/docu...
4,E,5,412833,4,4,3,4,4,4,5,...,20%,WRI/BMC summary based on satellite imagery 201...,WRI India green-cover study shared by BMC/MCAP...,https://indianexpress.com/article/cities/mumba...,Reported percentage,58028,2,Published green cover 20%; score 2 (low),"BMC Environment Status Report 2023–24, Table 8.1",https://portal.mcgm.gov.in/irj/go/km/docs/docu...
5,F_North,7,555328,4,4,3,4,3,4,2,...,Not found in cited public summary,WRI/BMC summary based on satellite imagery 201...,WRI India green-cover study shared by BMC/MCAP...,https://indianexpress.com/article/cities/mumba...,Percentage unavailable in accessible summary; ...,184837,4,Proxy only: BMC 2023–24 ward tree count quinti...,"BMC Environment Status Report 2023–24, Table 8.1",https://portal.mcgm.gov.in/irj/go/km/docs/docu...
6,F_South,6,378913,3,3,3,3,2,2,3,...,Not found in cited public summary,WRI/BMC summary based on satellite imagery 201...,WRI India green-cover study shared by BMC/MCAP...,https://indianexpress.com/article/cities/mumba...,Percentage unavailable in accessible summary; ...,87240,3,Proxy only: BMC 2023–24 ward tree count quinti...,"BMC Environment Status Report 2023–24, Table 8.1",https://portal.mcgm.gov.in/irj/go/km/docs/docu...
7,G_North,9,628812,4,4,4,4,3,4,2,...,Not found in cited public summary,WRI/BMC summary based on satellite imagery 201...,WRI India green-cover study shared by BMC/MCAP...,https://indianexpress.com/article/cities/mumba...,Percentage unavailable in accessible summary; ...,96620,3,Proxy only: BMC 2023–24 ward tree count quinti...,"BMC Environment Status Report 2023–24, Table 8.1",https://portal.mcgm.gov.in/irj/go/km/docs/docu...
8,G_South,8,396524,4,4,4,3,2,3,3,...,Not found in cited public summary,WRI/BMC summary based on satellite imagery 201...,WRI India green-cover study shared by BMC/MCAP...,https://indianexpress.com/article/cities/mumba...,Percentage unavaila

In [ ]:
WARD_COORDS = {
    1:  ("A",        18.9320, 72.8347),
    2:  ("B",        18.9550, 72.8350),
    3:  ("C",        18.9470, 72.8310),
    4:  ("D",        18.9560, 72.8150),
    5:  ("E",        18.9790, 72.8340),
    6:  ("F_South",  19.0030, 72.8420),
    7:  ("F_North",  19.0180, 72.8560),
    8:  ("G_South",  19.0050, 72.8250),
    9:  ("G_North",  19.0170, 72.8470),
    10: ("H_West",   19.0540, 72.8260),
    11: ("H_East",   19.0650, 72.8430),
    12: ("K_West",   19.1190, 72.8460),
    13: ("K_East",   19.1130, 72.8670),
    14: ("L",        19.0720, 72.8820),
    15: ("M_West",   19.0630, 72.8950),
    16: ("M_East",   19.0450, 72.9100),
    17: ("N",        19.0860, 72.9080),
    18: ("P_South",  19.1650, 72.8500),
    19: ("P_North",  19.1870, 72.8350),
    20: ("R_South",  19.2100, 72.8500),
    21: ("R_Central",19.2300, 72.8570),
    22: ("R_North",  19.2450, 72.8500),
    23: ("S",        19.1300, 72.9050),
    24: ("T",        19.1750, 72.9250)
}
def fetch_weather_for_day(date_str):
    rows = []
    for ward_id, (ward_name, lat, lon) in WARD_COORDS.items():
        url = "https://archive-api.open-meteo.com/v1/archive"
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": date_str,
            "end_date": date_str,
            "daily": [
                "temperature_2m_mean",
                "temperature_2m_max",
                "temperature_2m_min",
                "relative_humidity_2m_mean",
                "wind_speed_10m_mean",
                "shortwave_radiation_sum"
            ],"timezone": "Asia/Kolkata"
        }
        for attempt in range(5):
            try:
                response = requests.get(
                    url,
                    params=params,
                    timeout=30
                )
                if response.status_code == 200:
                    data = response.json()["daily"]
                    rows.append({
                        "date": date_str,
                        "temp_mean_c": data["temperature_2m_mean"][0],
                        "temp_max_c": data["temperature_2m_max"][0],
                        "temp_min_c": data["temperature_2m_min"][0],
                        "humidity_pct": data["relative_humidity_2m_mean"][0],
                        "wind_speed_ms": data["wind_speed_10m_mean"][0],
                        "solar_radiation_kwh_m2":
                            data["shortwave_radiation_sum"][0],
                        "ward_id": ward_id,
                        "ward_name": ward_name
                    })
                    break
                elif response.status_code == 429:
                    wait = 10 * (attempt + 1)
                    print(f"Rate limited for {ward_name}."f"Retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    print(f"Failed {ward_name}: "f"HTTP {response.status_code}")
                    break

            except Exception as e:
                print(f"Error for {ward_name}: {e}")
                time.sleep(5)
        time.sleep(0.5)
    return pd.DataFrame(rows)

def update_weather_database(csv_file="ward_ml_base.csv"):
    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        df["date"] = pd.to_datetime(df["date"])

        anchor_date = df["date"].max().normalize()
        target_date = (datetime.now() - timedelta(days=1))
        target_date = pd.Timestamp(target_date).normalize()
        if anchor_date >= target_date:
            print(f"✓ Anchor date: {anchor_date.date()}")
            print("✓ Weather database is up to date.")
            return df
        start_date = anchor_date + pd.Timedelta(days=1)
        print(f"Anchor date: {anchor_date.date()}")
        print(f"Downloading missing data:"f"{start_date.date()} → {target_date.date()}")
        all_data = []
        current_date = start_date
        while current_date <= target_date:
            date_str = current_date.strftime("%Y-%m-%d")
            print(f"Downloading {date_str}...")
            day_data = fetch_weather_for_day(date_str)
            if len(day_data) > 0:
                all_data.append(day_data)
            current_date += pd.Timedelta(days=1)
        if all_data:
            new_data = pd.concat(all_data,ignore_index=True)
            new_data["date"] = pd.to_datetime(new_data["date"])
            df = pd.concat([df, new_data],ignore_index=True)
    else:
        print("⚠ ward_ml_base.csv not found.")
        print("Downloading historical weather database...")
        start_date = pd.Timestamp("2022-01-01")
        target_date = pd.Timestamp(datetime.now() - timedelta(days=1)).normalize()
        all_data = []
        for ward_id, (ward_name, lat, lon) in WARD_COORDS.items():
            print(f"Downloading Ward {ward_name} "f"({ward_id}/24)...")
            url = "https://archive-api.open-meteo.com/v1/archive"
            params = {
                "latitude": lat,
                "longitude": lon,
                "start_date": start_date.strftime("%Y-%m-%d"),
                "end_date": target_date.strftime("%Y-%m-%d"),
                "daily": [
                    "temperature_2m_mean",
                    "temperature_2m_max",
                    "temperature_2m_min",
                    "relative_humidity_2m_mean",
                    "wind_speed_10m_mean",
                    "shortwave_radiation_sum"
                ],"timezone": "Asia/Kolkata"}

            for attempt in range(5):
                try:
                    response = requests.get(
                        url,
                        params=params,
                        timeout=60
                    )
                    if response.status_code == 200:
                        data = response.json()["daily"]
                        ward_data = pd.DataFrame({
                            "date": data["time"],
                            "temp_mean_c":
                                data["temperature_2m_mean"],
                            "temp_max_c":
                                data["temperature_2m_max"],
                            "temp_min_c":
                                data["temperature_2m_min"],
                            "humidity_pct":
                                data["relative_humidity_2m_mean"],
                            "wind_speed_ms":
                                data["wind_speed_10m_mean"],
                            "solar_radiation_kwh_m2":
                                data["shortwave_radiation_sum"],
                            "ward_id": ward_id,
                            "ward_name": ward_name
                        })
                        all_data.append(ward_data)
                        print(f"✓ Ward {ward_name}: "f"{len(ward_data)} days")
                        break
                    elif response.status_code == 429:
                        wait = 10 * (attempt + 1)
                        print(f"Rate limited for {ward_name}. "f"Retrying in {wait}s...")
                        time.sleep(wait)
                    else:
                        print(f"Failed {ward_name}: "f"HTTP {response.status_code}")
                        break
                except Exception as e:
                    print(f"Error for {ward_name}: {e}")
                    time.sleep(5)

    if not all_data and not os.path.exists(csv_file):
        raise RuntimeError("No weather data could be downloaded.")
    if all_data:
        new_df = pd.concat(all_data,ignore_index=True)
        if os.path.exists(csv_file):
            df = pd.concat([df, new_df],ignore_index=True)
        else:
            df = new_df
    df["date"] = pd.to_datetime(df["date"])
    df = (df.drop_duplicates(subset=["date", "ward_id"],keep="last").sort_values(["date", "ward_id"])
    .reset_index(drop=True))

    df.to_csv(csv_file,index=False)

    print()
    print("\nWEATHER DATABASE UPDATED\n")
    print(f"File: {csv_file}")
    print(f"Rows: {len(df):,}")
    print(f"Wards: {df['ward_id'].nunique()}")
    print(
        f"Date range: "
        f"{df['date'].min().date()} → "
        f"{df['date'].max().date()}"
    )
    return df
ward_ml_base = update_weather_database("ward_ml_base.csv")

⚠ ward_ml_base.csv not found.
✓ Ward A: 1763 days
✓ Ward B: 1763 days
✓ Ward C: 1763 days
✓ Ward D: 1763 days
✓ Ward E: 1763 days
✓ Ward F_South: 1763 days
✓ Ward F_North: 1763 days
✓ Ward G_South: 1763 days
Rate limited for G_North. Retrying in 10s...
Rate limited for G_North. Retrying in 20s...
✓ Ward G_North: 1763 days
✓ Ward H_West: 1763 days
✓ Ward H_East: 1763 days
✓ Ward K_West: 1763 days
✓ Ward K_East: 1763 days
✓ Ward L: 1763 days
✓ Ward M_West: 1763 days
✓ Ward M_East: 1763 days
Rate limited for N. Retrying in 10s...
Rate limited for N. Retrying in 20s...
Rate limited for N. Retrying in 30s...
✓ Ward N: 1763 days
✓ Ward P_South: 1763 days
✓ Ward P_North: 1763 days
✓ Ward R_South: 1763 days
✓ Ward R_Central: 1763 days
✓ Ward R_North: 1763 days
✓ Ward S: 1763 days
✓ Ward T: 1763 days


WEATHER DATABASE UPDATED

File: ward_ml_base.csv
Rows: 42,312
Wards: 24
Date range: 2022-01-01 → 2026-10-29


In [ ]:
df = pd.read_csv("ward_ml_base.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["ward_id", "date"]).reset_index(drop=True)
print("Loaded:", df.shape)
print("Wards:", df["ward_name"].nunique())
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())

def calculate_heat_index(T, RH):
    T_f = T * 9/5 + 32
    HI_f = 0.5 * (T_f + 61.0 + ((T_f - 68.0) * 1.2) + (RH * 0.094))
    HI_roth_f = (
        -42.379
        + 2.04901523 * T_f
        + 10.14333127 * RH
        - 0.22475541 * T_f * RH
        - 0.00683783 * T_f**2
        - 0.05481717 * RH**2
        + 0.00122874 * T_f**2 * RH
        + 0.00085282 * T_f * RH**2
        - 0.00000199 * T_f**2 * RH**2)
    use_roth = ((T_f >= 80) &(RH >= 40))
    HI_f = np.where(use_roth, HI_roth_f, HI_f)
    return (HI_f - 32) * 5/9

df["heat_index_c"] = calculate_heat_index(df["temp_mean_c"],df["humidity_pct"])

def wet_bulb_stull(T, RH):
    return (
        T * np.arctan(0.151977 * np.sqrt(RH + 8.313659))
        + np.arctan(T + RH)
        - np.arctan(RH - 1.676331)
        + 0.00391838 * RH**1.5
        * np.arctan(0.023101 * RH)
        - 4.686035
    )
df["wet_bulb_c"] = wet_bulb_stull(df["temp_mean_c"],df["humidity_pct"])
df["utci_c"] = (df["temp_mean_c"]+ 0.33 * (df["humidity_pct"] / 100 * 20 - 10)- 0.70 * df["wind_speed_ms"]
                + 0.02 * df["solar_radiation_kwh_m2"])
solar = df["solar_radiation_kwh_m2"].clip(lower=0)
df["globe_temp_c"]=(df["temp_mean_c"]+0.15 * solar)
df["wbgt_c"] = (0.7 * df["wet_bulb_c"] +0.2 * df["globe_temp_c"]+0.1*df["temp_mean_c"])
df["month"] = df["date"].dt.month

baseline = (
    df[df["date"].dt.year <= 2023]
    .groupby(["ward_id", "month"])["temp_mean_c"]
    .mean()
    .rename("monthly_temp_baseline")
    .reset_index()
)
df = df.merge(baseline,on=["ward_id", "month"],how="left")

df["temperature_anomaly_c"] = (df["temp_mean_c"]-df["monthly_temp_baseline"])

grouped = df.groupby("ward_id", group_keys=False)

df["temp_lag_1d"] = grouped["temp_mean_c"].shift(1)
df["temp_lag_3d"] = grouped["temp_mean_c"].shift(3)
df["temp_lag_7d"] = grouped["temp_mean_c"].shift(7)

df["humidity_lag_1d"] = grouped["humidity_pct"].shift(1)
df["humidity_lag_3d"] = grouped["humidity_pct"].shift(3)
df["humidity_lag_7d"] = grouped["humidity_pct"].shift(7)
df["target_wbgt_5d"] = grouped["wbgt_c"].shift(-5)

df = df.drop(
    columns=[
        "month",
        "monthly_temp_baseline",
        "wet_bulb_c",
        "globe_temp_c"
    ]
)

required_features = [
    "temp_mean_c",
    "temp_max_c",
    "temp_min_c",
    "humidity_pct",
    "wind_speed_ms",
    "solar_radiation_kwh_m2",
    "heat_index_c",
    "wbgt_c",
    "utci_c",
    "temperature_anomaly_c",
    "temp_lag_1d",
    "temp_lag_3d",
    "temp_lag_7d",
    "humidity_lag_1d",
    "humidity_lag_3d",
    "humidity_lag_7d",
    "target_wbgt_5d"
]

df = df.dropna(subset=required_features).reset_index(drop=True)
print("\nFINAL WEATHER ML DATASET\n")
print("Shape:", df.shape)
print("Wards:", df["ward_name"].nunique())
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("\nColumns:")
print(df.columns.tolist())
print("\nMissing values:")
print(df[required_features].isna().sum())
print("\nWBGT statistics:")
print(df["wbgt_c"].describe())
print("\nTarget statistics:")
print(df[["target_wbgt_5d"]].describe())
df.to_csv("ward_ml_features.csv",index=False)
print("\nSaved: ward_ml_features.csv")

Loaded: (42312, 9)
Wards: 24
Date range: 2022-01-01 to 2026-10-29

FINAL WEATHER ML DATASET

Shape: (42024, 20)
Wards: 24
Date range: 2022-01-08 to 2026-10-24

Columns:
['date', 'temp_mean_c', 'temp_max_c', 'temp_min_c', 'humidity_pct', 'wind_speed_ms', 'solar_radiation_kwh_m2', 'ward_id', 'ward_name', 'heat_index_c', 'utci_c', 'wbgt_c', 'temperature_anomaly_c', 'temp_lag_1d', 'temp_lag_3d', 'temp_lag_7d', 'humidity_lag_1d', 'humidity_lag_3d', 'humidity_lag_7d', 'target_wbgt_5d']

Missing values:
temp_mean_c               0
temp_max_c                0
temp_min_c                0
humidity_pct              0
wind_speed_ms             0
solar_radiation_kwh_m2    0
heat_index_c              0
wbgt_c                    0
utci_c                    0
temperature_anomaly_c     0
temp_lag_1d               0
temp_lag_3d               0
temp_lag_7d               0
humidity_lag_1d           0
humidity_lag_3d           0
humidity_lag_7d           0
target_wbgt_5d            0
dtype: int64

WBGT sta

In [ ]:
df = pd.read_csv("ward_ml_features.csv")
df["date"] = pd.to_datetime(df["date"])
print("Dataset:", df.shape)
feature_columns = [
    "temp_mean_c",
    "temp_max_c",
    "temp_min_c",
    "humidity_pct",
    "wind_speed_ms",
    "solar_radiation_kwh_m2",
    "heat_index_c",
    "wbgt_c",
    "utci_c",
    "temperature_anomaly_c",
    "temp_lag_1d",
    "temp_lag_3d",
    "temp_lag_7d",
    "humidity_lag_1d",
    "humidity_lag_3d",
    "humidity_lag_7d"
]

targets = {"5d": "target_wbgt_5d"}

train = df[(df["date"].dt.year >= 2022) &(df["date"].dt.year <= 2023)].copy()
val = df[df["date"].dt.year == 2024].copy()
test = df[df["date"].dt.year >= 2025].copy()

print("\nSplit:")
print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

models = {}
results = []

for horizon, target in targets.items():

    print(f"\nTRAINING {horizon.upper()} MODEL\n")
    X_train = train[feature_columns]
    y_train = train[target]
    X_val = val[feature_columns]
    y_val = val[target]
    X_test = test[feature_columns]
    y_test = test[target]
    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train,y_train,eval_set=[(X_val, y_val)],verbose=False)

    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)

    val_mae = mean_absolute_error(y_val, val_pred)
    val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    val_r2 = r2_score(y_val, val_pred)

    test_mae = mean_absolute_error(y_test, test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    test_r2 = r2_score(y_test, test_pred)

    print("\nValidation:")
    print(f"MAE  : {val_mae:.4f} °C")
    print(f"RMSE : {val_rmse:.4f} °C")
    print(f"R²   : {val_r2:.4f}")
    print("\nTest:")
    print(f"MAE  : {test_mae:.4f} °C")
    print(f"RMSE : {test_rmse:.4f} °C")
    print(f"R²   : {test_r2:.4f}")

    models[horizon] = model

    results.append({
        "horizon": horizon,
        "val_mae": val_mae,
        "val_rmse": val_rmse,
        "val_r2": val_r2,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "test_r2": test_r2
    })

results_df = pd.DataFrame(results)

print("\nFINAL MODEL RESULTS\n")
print(results_df.to_string(index=False))
print("\nPERSISTENCE BASELINE\n")


baseline_results = []
for horizon, target in targets.items():
    y_true = test[target]
    baseline_pred = test["wbgt_c"]
    mae = mean_absolute_error(y_true, baseline_pred)
    rmse = np.sqrt(mean_squared_error(y_true, baseline_pred))
    r2 = r2_score(y_true, baseline_pred)
    baseline_results.append({
        "horizon": horizon,"mae": mae,
        "rmse": rmse,
        "r2": r2
    })

    print(
        f"{horizon}: "
        f"MAE={mae:.4f}, "
        f"RMSE={rmse:.4f}, "
        f"R²={r2:.4f}"
    )

for horizon, model in models.items():
    filename = f"xgb_wbgt_{horizon}.joblib"
    joblib.dump(model,filename)
    print(f"Saved: {filename}")

results_df.to_csv("xgb_model_results.csv",index=False)
print("\nSaved: xgb_model_results.csv")

Dataset: (42024, 20)

Split:
Train: (17352, 20)
Validation: (8784, 20)
Test: (15888, 20)

TRAINING 5D MODEL


Validation:
MAE  : 0.9689 °C
RMSE : 1.2979 °C
R²   : 0.7573

Test:
MAE  : 0.8509 °C
RMSE : 1.1261 °C
R²   : 0.7685

FINAL MODEL RESULTS

horizon  val_mae  val_rmse   val_r2  test_mae  test_rmse  test_r2
     5d 0.968882  1.297927 0.757344   0.85092   1.126068 0.768478

PERSISTENCE BASELINE

5d: MAE=0.8528, RMSE=1.1491, R²=0.7589
Saved: xgb_wbgt_5d.joblib

Saved: xgb_model_results.csv


In [ ]:
ml = pd.read_csv("ward_ml_features.csv")
ml["date"] = pd.to_datetime(ml["date"])
print("Vulnerability data:", vulnerability.shape)
print("ML data:", ml.shape)

ml_final = ml.merge(vulnerability,on=["ward_id", "ward_name"],how="left")

print("\nMERGED DATASET\n")
print("Shape:", ml_final.shape)
print("Wards:", ml_final["ward_name"].nunique())

check_columns = [
    "population_2025",
    "vulnerability_index_category",
    "exposure_index_category",
    "green_cover_score_1_to_5"
]

print("\nMissing vulnerability values:")
print(ml_final[check_columns].isna().sum())

print("\nSample:")
print(ml_final[[
            "date",
            "ward_id",
            "ward_name",
            "wbgt_c",
            "population_2025",
            "vulnerability_index_category",
            "exposure_index_category",
            "green_cover_score_1_to_5"
        ]].head(10).to_string(index=False))

ml_final.to_csv("ward_ml_final.csv",index=False)
print("\n✓ Saved: ward_ml_final.csv")

Vulnerability data: (24, 32)
ML data: (42024, 20)

MERGED DATASET

Shape: (42024, 50)
Wards: 24

Missing vulnerability values:
population_2025                 0
vulnerability_index_category    0
exposure_index_category         0
green_cover_score_1_to_5        0
dtype: int64

Sample:
      date  ward_id ward_name    wbgt_c  population_2025  vulnerability_index_category  exposure_index_category  green_cover_score_1_to_5
2022-01-08        1         A 21.647293           194120                             2                        3                         2
2022-01-09        1         A 18.886856           194120                             2                        3                         2
2022-01-10        1         A 16.957041           194120                             2                        3                         2
2022-01-11        1         A 18.190797           194120                             2                        3                         2
2022-01-12        1      

In [ ]:
df = pd.read_csv("ward_ml_base.csv")
df["date"] = pd.to_datetime(df["date"])
df = (df.sort_values(["ward_id", "date"]).reset_index(drop=True))
print("Loaded raw data:", df.shape)
print("Historical date range:",df["date"].min().date(),"to",df["date"].max().date())
FORECAST_DATE = pd.Timestamp(datetime.now().date())
print("Operational anchor date:",FORECAST_DATE.date())
latitudes = ",".join(str(WARD_COORDS[i][1])for i in sorted(WARD_COORDS))
longitudes = ",".join(str(WARD_COORDS[i][2])for i in sorted(WARD_COORDS))
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": latitudes,
    "longitude": longitudes,
    "daily": (
        "temperature_2m_mean,"
        "temperature_2m_max,"
        "temperature_2m_min,"
        "relative_humidity_2m_mean,"
        "wind_speed_10m_mean,"
        "shortwave_radiation_sum"),
    "timezone": "Asia/Kolkata",
    "forecast_days": 1
}
print("\nFetching today's operational weather...")
response = requests.get(url,params=params,timeout=60)
response.raise_for_status()
weather_json = response.json()
if isinstance(weather_json, dict):
    weather_json = [weather_json]
operational_rows = []
for index, ward_id in enumerate(sorted(WARD_COORDS)):
    ward_name = WARD_COORDS[ward_id][0]
    daily = weather_json[index]["daily"]
    operational_rows.append({
        "date": FORECAST_DATE,
        "temp_mean_c": daily["temperature_2m_mean"][0],
        "temp_max_c": daily["temperature_2m_max"][0],
        "temp_min_c": daily["temperature_2m_min"][0],
        "humidity_pct": daily["relative_humidity_2m_mean"][0],
        "wind_speed_ms": (daily["wind_speed_10m_mean"][0]),
        "solar_radiation_kwh_m2": (daily["shortwave_radiation_sum"][0] ),
        "ward_id": ward_id,
        "ward_name": ward_name
    })

operational = pd.DataFrame(operational_rows)
print("Operational rows:", len(operational))
if len(operational) != 24:
    raise ValueError(f"Expected 24 operational rows,"f"found {len(operational)}")
operational["heat_index_c"] = calculate_heat_index(operational["temp_mean_c"],operational["humidity_pct"])
wet_bulb = wet_bulb_stull(operational["temp_mean_c"],operational["humidity_pct"])
solar = operational["solar_radiation_kwh_m2"].clip(lower=0)
globe_temp = (operational["temp_mean_c"]+0.15*solar)
operational["wbgt_c"] = (0.7 * wet_bulb+ 0.2 * globe_temp+ 0.1 * operational["temp_mean_c"])
operational["utci_c"] = (operational["temp_mean_c"]+ 0.33 * (operational["humidity_pct"] / 100 * 20 - 10)
    - 0.70 * operational["wind_speed_ms"]+ 0.02 * operational["solar_radiation_kwh_m2"])
df["month"] = df["date"].dt.month
baseline = (
    df[df["date"].dt.year <= 2023]
    .groupby(["ward_id", "month"])["temp_mean_c"]
    .mean()
    .rename("monthly_temp_baseline")
    .reset_index())
operational["month"] = operational["date"].dt.month
operational = operational.merge(baseline,on=["ward_id", "month"],how="left")
operational["temperature_anomaly_c"] = (operational["temp_mean_c"]- operational["monthly_temp_baseline"])
operational.drop(columns=["month", "monthly_temp_baseline"],inplace=True)
for lag in [1, 3, 7]:
    lag_date = (FORECAST_DATE-pd.Timedelta(days=lag))
    lag_data = df[df["date"] == lag_date][["ward_id", "temp_mean_c", "humidity_pct"]].copy()
    lag_data.rename(columns={"temp_mean_c": f"temp_lag_{lag}d","humidity_pct": f"humidity_lag_{lag}d"},inplace=True)
    operational = operational.merge( lag_data,on="ward_id",how="left")
FEATURE_COLS = [
    "temp_mean_c",
    "temp_max_c",
    "temp_min_c",
    "humidity_pct",
    "wind_speed_ms",
    "solar_radiation_kwh_m2",
    "heat_index_c",
    "wbgt_c",
    "utci_c",
    "temperature_anomaly_c",
    "temp_lag_1d",
    "temp_lag_3d",
    "temp_lag_7d",
    "humidity_lag_1d",
    "humidity_lag_3d",
    "humidity_lag_7d"
]
missing = [c for c in FEATURE_COLS if c not in operational.columns]
if missing:
    raise ValueError(f"Missing features: {missing}")
if operational[FEATURE_COLS].isna().any().any():
    print("\nMissing feature values:")
    print(operational[FEATURE_COLS].isna().sum())
    raise ValueError("Some operational feature values are missing.")
print("All operational features available")
model_5d = joblib.load("xgb_wbgt_5d.joblib")
print(" Models loaded")
X = operational[FEATURE_COLS]
operational["predicted_wbgt_5d"] = (model_5d.predict(X))
forecast_output = operational[
    [
        "ward_id",
        "ward_name",
        "temp_mean_c",
        "temp_max_c",
        "humidity_pct",
        "wind_speed_ms",
        "solar_radiation_kwh_m2",
        "wbgt_c",
        "utci_c",
        "heat_index_c",
        "predicted_wbgt_5d"
    ]].copy()
numeric_cols = [
    "temp_mean_c",
    "temp_max_c",
    "humidity_pct",
    "wind_speed_ms",
    "solar_radiation_kwh_m2",
    "wbgt_c",
    "utci_c",
    "heat_index_c",
    "predicted_wbgt_5d"
]
forecast_output[numeric_cols] = (forecast_output[numeric_cols].round(2))
print()
print("\nREAL-TIME OPERATIONAL WBGT FORECAST\n")
print("Anchor:", FORECAST_DATE.date())
print("+5d:",(FORECAST_DATE + pd.Timedelta(days=5)).date())
print()
display(
    forecast_output[
        [
            "ward_id",
            "ward_name",
            "predicted_wbgt_5d"
        ]].sort_values("ward_id"))

print()
print("\nFORECAST SUMMARY\n")
for horizon, col, days in [
    ("+5d", "predicted_wbgt_5d", 5)
]:
    forecast_date = (FORECAST_DATE+ pd.Timedelta(days=days))
    print()
    print(f"{horizon} — {forecast_date.date()}")
    print(f"Min : {forecast_output[col].min():.2f} °C")
    print(f"Mean: {forecast_output[col].mean():.2f} °C")
    print(f"Max : {forecast_output[col].max():.2f} °C")


forecast_output.to_csv("mumbai_operational_wbgt_forecast.csv",index=False)
print()
print("\n OPERATIONAL FORECAST SAVED\n")
print("File: mumbai_operational_wbgt_forecast.csv")

Loaded raw data: (42312, 9)
Historical date range: 2022-01-01 to 2026-10-29
Operational anchor date: 2026-10-30

Fetching today's operational weather...
Operational rows: 24
All operational features available
 Models loaded


REAL-TIME OPERATIONAL WBGT FORECAST

Anchor: 2026-10-30
+5d: 2026-11-04



,ward_id,ward_name,predicted_wbgt_5d
0,1,A,25.780001
1,2,B,25.520000
2,3,C,25.780001
3,4,D,25.340000
4,5,E,25.510000
5,6,F_South,25.780001
6,7,F_North,25.780001
7,8,G_South,25.780001
8,9,G_North,25.780001
9,10,H_West,24.889999




FORECAST SUMMARY


+5d — 2026-11-04
Min : 24.89 °C
Mean: 25.32 °C
Max : 25.78 °C


 OPERATIONAL FORECAST SAVED

File: mumbai_operational_wbgt_forecast.csv


In [ ]:
FORECAST_FILE = "mumbai_operational_wbgt_forecast.csv"
forecast = pd.read_csv(FORECAST_FILE)
print("Forecast data:", forecast.shape)
print("Vulnerability data:", vulnerability.shape)
forecast["ward_id"] = forecast["ward_id"].astype(int)
vulnerability["ward_id"] = vulnerability["ward_id"].astype(int)
VULNERABILITY_COLS = [
    "population_density_category",
    "household_density_category",
    "socially_weaker_section_category"
]
EXPOSURE_COLS = [
    "marginal_workers_category",
    "illiteracy_rate_category",
    "dilapidated_household_category",
    "no_electric_connection_category",
    "untreated_water_quality_category",
    "distant_water_source_category"
]
GREEN_COVER_COL = "green_cover_score_1_to_5"
required_cols = (
    ["ward_id", "ward_name"]+ VULNERABILITY_COLS+ EXPOSURE_COLS+ [GREEN_COVER_COL])
missing = [
    col for col in required_cols
    if col not in vulnerability.columns
]
if missing:
    raise ValueError(f"Missing columns: {missing}")
for col in VULNERABILITY_COLS + EXPOSURE_COLS:
    vulnerability[col] = pd.to_numeric(vulnerability[col],errors="coerce")
    if vulnerability[col].isna().any():
        raise ValueError(f"Invalid values in {col}")
vulnerability[GREEN_COVER_COL] = pd.to_numeric(
    vulnerability[GREEN_COVER_COL],
    errors="coerce")
if vulnerability[GREEN_COVER_COL].isna().any():
    raise ValueError(f"Invalid values in {GREEN_COVER_COL}")
if not vulnerability[GREEN_COVER_COL].between(1, 5).all():
    raise ValueError(
        f"{GREEN_COVER_COL} must contain scores from 1 to 5.")
vulnerability["V"] = (vulnerability[VULNERABILITY_COLS].sub(1).div(4).mean(axis=1))
vulnerability["E"] = (vulnerability[EXPOSURE_COLS].sub(1).div(4).mean(axis=1))
vulnerability["G"]=(6 - vulnerability[GREEN_COVER_COL]) / 5
risk_df = forecast.merge(
    vulnerability[
        [
            "ward_id",
            "ward_name",
            "V",
            "E",
            "G",
            GREEN_COVER_COL
        ]
    ],
    on="ward_id",
    how="left",
    suffixes=("_forecast", "_vulnerability"),
    validate="many_to_one"
)

if len(risk_df) != 24:
    raise ValueError(f"Expected 24 rows, got {len(risk_df)}")

if (
    risk_df["V"].isna().any()
    or risk_df["E"].isna().any()
    or risk_df["G"].isna().any()
):
    raise ValueError("Missing vulnerability, exposure, or green-cover values after merge.")
def thermal_hazard(wbgt):
    return np.clip(
        (wbgt - 21.0) / 8.0,
        0,
        1
    )
HORIZONS = [("5d", "predicted_wbgt_5d", 5)]
if "forecast_date" in forecast.columns:
    anchor_date = pd.to_datetime(
        forecast["forecast_date"]).max().normalize()
else:
    anchor_date = pd.Timestamp.today().normalize()
print("\nOPERATIONAL ANCHOR\n")
print("Anchor date:", anchor_date.strftime("%Y-%m-%d"))
all_results = []
for horizon, wbgt_col, days_ahead in HORIZONS:
    temp = risk_df.copy()
    temp["predicted_wbgt"] = temp[wbgt_col]
    temp["thermal_hazard"] = thermal_hazard(temp["predicted_wbgt"])
    temp["risk_amplification"] = (0.50+ 0.25 * temp["V"]+ 0.15 * temp["E"]+ 0.10 * temp["G"])
    temp["HMRI"] = (temp["thermal_hazard"]* temp["risk_amplification"]* 100)
    temp["forecast_horizon"] = horizon
    temp["forecast_date"] = (anchor_date + pd.Timedelta(days=days_ahead))
    all_results.append(temp)
hmri_results = pd.concat(all_results,ignore_index=True)
hmri_results["forecast_date"] = (hmri_results["forecast_date"].dt.strftime("%Y-%m-%d"))
hmri_results["predicted_wbgt"] = (hmri_results["predicted_wbgt"].round(2))
hmri_results["HMRI"] = (hmri_results["HMRI"].round(1))

hmri_results["V"] = hmri_results["V"].round(3)
hmri_results["E"] = hmri_results["E"].round(3)
hmri_results["G"] = hmri_results["G"].round(3)

hmri_results["thermal_hazard"] = (hmri_results["thermal_hazard"].round(3))
hmri_results["risk_amplification"] = (hmri_results["risk_amplification"].round(3))

final_hmri = hmri_results[
    [
        "forecast_date",
        "forecast_horizon",
        "ward_id",
        "ward_name_forecast",
        "predicted_wbgt",
        "V",
        "E",
        "G",
        GREEN_COVER_COL,
        "thermal_hazard",
        "risk_amplification",
        "HMRI"
    ]].copy()
final_hmri = final_hmri.rename(columns={"ward_name_forecast": "ward_name"})

final_hmri = (final_hmri.sort_values(["forecast_date", "ward_id"]).reset_index(drop=True))
print("\nFINAL HMRI FORECAST\n")
for date in final_hmri["forecast_date"].unique():
    print()
    print(f"FORECAST DATE: {date}")
    display(
        final_hmri[
            final_hmri["forecast_date"] == date
        ][
            [
                "ward_id",
                "ward_name",
                "predicted_wbgt",
                GREEN_COVER_COL,
                "HMRI"
            ]
        ]
    )
final_hmri.to_csv("mumbai_operational_hmri.csv",index=False)
print()
print("=" * 70)
print("✓ HMRI SAVED")
print("=" * 70)
print("File: mumbai_operational_hmri.csv")


Forecast data: (24, 11)
Vulnerability data: (24, 32)

OPERATIONAL ANCHOR

Anchor date: 2026-10-30

FINAL HMRI FORECAST


FORECAST DATE: 2026-11-04


,ward_id,ward_name,predicted_wbgt,green_cover_score_1_to_5,HMRI
0,1,A,25.78,2,44.5
1,2,B,25.52,1,47.1
2,3,C,25.78,1,48.4
3,4,D,25.34,3,39.4
4,5,E,25.51,2,48.1
5,6,F_South,25.78,3,43.9
6,7,F_North,25.78,4,46.0
7,8,G_South,25.78,3,48.0
8,9,G_North,25.78,3,48.8
9,10,H_West,24.89,2,36.3



✓ HMRI SAVED
File: mumbai_operational_hmri.csv


In [ ]:
files.download('xgb_wbgt_5d.joblib')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Integrated Step 11 — Budget Optimization + Ward-Level Backend CSV

This section runs **after the HMRI cell** and uses `final_hmri` produced by the existing pipeline. It keeps the WBGT forecasting and HMRI formulas unchanged.

The optimizer:
- outputs every ward from the vulnerability CSV, including wards with ₹0 allocation;
- uses HMRI/WBGT, vulnerability, exposure, and green-cover disadvantage for priority;
- allocates quantities of interventions within the budget;
- reduces greenery-action priority when green-cover score is high;
- puts past completed actions and new recommendations in the existing `action` field only (no separate action-history description column);
- writes one row per ward to `mumbai_heatshield_backend_output.csv`;
- persists action history in `mumbai_heatshield_action_history.csv` so the next run can avoid repeating the same action type.

**Prototype assumption:** per your requested scenario, recommendations made on a run are treated as completed for the next run. This is not evidence that work actually occurred. For real deployment, replace this simulated status with implementation-team confirmation.

**Cost and impact values below are placeholders** and must be replaced with locally sourced intervention costs/effectiveness before treating predicted reductions as real-world estimates.


## 11.1 Set budget and intervention assumptions

In [ ]:
# Set this to the actual budget for the forecast date.
# Default budget: ₹50 lakh
DEFAULT_BUDGET_INR = 5_000_000

# Illustrative prototype assumptions only: replace with sourced costs and impact estimates.
# max_units_per_ward controls the maximum number of units of each action in one ward.
INTERVENTIONS = [
    {"key": "water_kiosk", "label": "water kiosks", "unit": "kiosk",
     "cost_inr": 100_000, "hmri_reduction_fraction": 0.035,
     "max_units_per_ward": 4, "type": "water"},
    {"key": "shade_station", "label": "shaded rest stations", "unit": "station",
     "cost_inr": 250_000, "hmri_reduction_fraction": 0.070,
     "max_units_per_ward": 2, "type": "shade"},
    {"key": "cool_roof", "label": "cool-roof packages", "unit": "package",
     "cost_inr": 200_000, "hmri_reduction_fraction": 0.050,
     "max_units_per_ward": 3, "type": "cool_roof"},
    {"key": "cooling_centre", "label": "cooling-centre support units", "unit": "unit",
     "cost_inr": 500_000, "hmri_reduction_fraction": 0.110,
     "max_units_per_ward": 1, "type": "cooling"},
    {"key": "greenery", "label": "shade-tree/greenery packages", "unit": "package",
     "cost_inr": 150_000, "hmri_reduction_fraction": 0.040,
     "max_units_per_ward": 3, "type": "greenery"},
]

ACTION_HISTORY_FILE = "mumbai_heatshield_action_history.csv"
BACKEND_OUTPUT_CSV = "mumbai_heatshield_backend_output.csv"
BACKEND_OUTPUT_JSON = "mumbai_heatshield_backend_output.json"

# Green score 1 = low green cover; score 5 = high green cover.
# Greenery multiplier falls linearly from 1.0 at score 1 to 0.2 at score 5.
GREENERY_PRIORITY_MIN = 0.20
GREENERY_PRIORITY_MAX = 1.00


## 11.2 Build the 24-ward input and validate it

In [ ]:
# final_hmri is created in the preceding HMRI cell.
# Ensure we use the full vulnerability ward list as the left-hand side so unfunded wards remain.
required_hmri_cols = {
    "ward_id", "ward_name", "predicted_wbgt", "HMRI", "forecast_date"
}
missing_hmri = required_hmri_cols - set(final_hmri.columns)
if missing_hmri:
    raise ValueError(f"final_hmri is missing required columns: {sorted(missing_hmri)}")

vuln_opt = vulnerability.copy()
vuln_opt["ward_id"] = pd.to_numeric(vuln_opt["ward_id"], errors="coerce").astype("Int64")
hmri_opt = final_hmri.copy()
hmri_opt["ward_id"] = pd.to_numeric(hmri_opt["ward_id"], errors="coerce").astype("Int64")

# The HMRI file may contain one forecast horizon; fail rather than silently duplicate wards.
if hmri_opt["ward_id"].duplicated().any():
    raise ValueError("final_hmri has multiple rows per ward. Select one forecast horizon before optimization.")

if len(vuln_opt) != 24:
    raise ValueError(f"Expected 24 wards in vulnerability data, found {len(vuln_opt)}.")
if vuln_opt["ward_id"].isna().any() or vuln_opt["ward_id"].duplicated().any():
    raise ValueError("Vulnerability ward IDs must be numeric and unique.")

FORECAST_DATE = str(pd.to_datetime(hmri_opt["forecast_date"]).max().date())

# Require the input forecast to cover all 24 wards for a valid allocation run.
forecast_ids = set(hmri_opt["ward_id"].dropna().astype(int))
vulnerability_ids = set(vuln_opt["ward_id"].dropna().astype(int))
missing_forecast_ids = sorted(vulnerability_ids - forecast_ids)
if missing_forecast_ids:
    print("WARNING: No forecast for ward IDs:", missing_forecast_ids,
          "— those wards will remain in the CSV with ₹0 allocation.")

print("Forecast date:", FORECAST_DATE)
print("Wards in vulnerability data:", len(vuln_opt))
print("Wards with forecast:", len(forecast_ids))
print("Budget: ₹{:,.0f}".format(BUDGET_INR))


Forecast date: 2026-11-04
Wards in vulnerability data: 24
Wards with forecast: 24
Budget: ₹5,000,000


## 11.3 Priority scores

In [ ]:
# Use the same ordinal 1–5 input categories already used by the HMRI pipeline.
# This score is a transparent allocation heuristic, not a published BMC formula.
for col in [
    "population_density_category", "household_density_category",
    "socially_weaker_section_category", "marginal_workers_category",
    "illiteracy_rate_category", "dilapidated_household_category",
    "no_electric_connection_category", "untreated_water_quality_category",
    "distant_water_source_category", "green_cover_score_1_to_5"
]:
    if col not in vuln_opt.columns:
        raise ValueError(f"Required vulnerability column not found: {col}")
    vuln_opt[col] = pd.to_numeric(vuln_opt[col], errors="coerce")

VULNERABILITY_COLS_OPT = [
    "population_density_category", "household_density_category",
    "socially_weaker_section_category"
]
EXPOSURE_COLS_OPT = [
    "marginal_workers_category", "illiteracy_rate_category",
    "dilapidated_household_category", "no_electric_connection_category",
    "untreated_water_quality_category", "distant_water_source_category"
]
if vuln_opt[VULNERABILITY_COLS_OPT + EXPOSURE_COLS_OPT + ["green_cover_score_1_to_5"]].isna().any().any():
    raise ValueError("Missing/non-numeric vulnerability or green-cover values.")

vuln_opt["V_opt"] = (vuln_opt[VULNERABILITY_COLS_OPT].sub(1).div(4).mean(axis=1)).clip(0, 1)
vuln_opt["E_opt"] = (vuln_opt[EXPOSURE_COLS_OPT].sub(1).div(4).mean(axis=1)).clip(0, 1)
vuln_opt["greenery_disadvantage_opt"] = ((5 - vuln_opt["green_cover_score_1_to_5"]) / 4).clip(0, 1)

forecast_cols = hmri_opt[["ward_id", "predicted_wbgt", "HMRI", "forecast_date"]].rename(
    columns={"HMRI": "predicted_hmri"}
)
optimizer_wards = vuln_opt.merge(forecast_cols, on="ward_id", how="left", validate="one_to_one")

optimizer_wards["wbgt_component_opt"] = ((optimizer_wards["predicted_wbgt"] - 25.0) / 10.0).clip(0, 1)
optimizer_wards["hmri_component_opt"] = (optimizer_wards["predicted_hmri"] / 100.0).clip(0, 1)

optimizer_wards["priority_score"] = 100 * (
    0.60 * optimizer_wards["hmri_component_opt"]
    + 0.20 * optimizer_wards["wbgt_component_opt"]
    + 0.10 * optimizer_wards["V_opt"]
    + 0.05 * optimizer_wards["E_opt"]
    + 0.05 * optimizer_wards["greenery_disadvantage_opt"]
)
optimizer_wards.loc[
    optimizer_wards["predicted_hmri"].isna() | optimizer_wards["predicted_wbgt"].isna(),
    "priority_score"
] = np.nan

display(optimizer_wards[[
    "ward_id", "ward_name", "predicted_wbgt", "predicted_hmri",
    "green_cover_score_1_to_5", "priority_score"
]].sort_values("priority_score", ascending=False, na_position="last"))


,ward_id,ward_name,predicted_wbgt,predicted_hmri,green_cover_score_1_to_5,priority_score
4,5,E,25.51,48.1,2,43.838333
2,3,C,25.78,48.4,1,43.725000
7,9,G_North,25.78,48.8,3,43.131667
1,2,B,25.52,47.1,1,43.050000
8,8,G_South,25.78,48.0,3,42.235000
5,7,F_North,25.78,46.0,4,39.160000
14,16,M_East,25.78,45.6,4,38.295000
0,1,A,25.78,44.5,2,38.051667
13,14,L,24.89,40.3,1,37.930000
6,6,F_South,25.78,43.9,3,37.066667


## 11.4 Load action history

In [ ]:
HISTORY_COLUMNS = [
    "forecast_date", "ward_id", "action_key", "action_label",
    "quantity", "unit_cost_inr", "total_cost_inr", "status"
]

def load_action_history(path=ACTION_HISTORY_FILE):
    if not os.path.exists(path):
        return pd.DataFrame(columns=HISTORY_COLUMNS)
    h = pd.read_csv(path)
    for col in HISTORY_COLUMNS:
        if col not in h.columns:
            h[col] = pd.NA
    h = h[HISTORY_COLUMNS].copy()
    h["ward_id"] = h["ward_id"].astype(str)
    h["quantity"] = pd.to_numeric(h["quantity"], errors="coerce").fillna(0).astype(int)
    h["forecast_date"] = h["forecast_date"].astype(str)
    return h

action_history = load_action_history()
print("Historical action records:", len(action_history))


Historical action records: 0


## 11.5 Allocate the budget

In [ ]:
def completed_quantity(history, ward_id, action_key):
    if history.empty:
        return 0
    mask = (
        history["ward_id"].astype(str).eq(str(int(ward_id)))
        & history["action_key"].astype(str).eq(str(action_key))
        & history["status"].astype(str).str.lower().eq("completed")
    )
    return int(pd.to_numeric(history.loc[mask, "quantity"], errors="coerce").fillna(0).sum())


def greenery_multiplier(green_score):
    if pd.isna(green_score):
        return 0.0
    score = float(green_score)
    return GREENERY_PRIORITY_MAX - ((score - 1.0) / 4.0) * (
        GREENERY_PRIORITY_MAX - GREENERY_PRIORITY_MIN
    )


def allocate_budget(wards, budget_inr, history, interventions=INTERVENTIONS):
    # Greedy marginal allocator: repeatedly funds the most valuable affordable unit.
    # It is a heuristic, not a proof of global optimality.
    budget_left = int(budget_inr)
    allocations = {int(wid): [] for wid in wards["ward_id"].dropna().astype(int)}
    candidates = []

    for _, row in wards.iterrows():
        if pd.isna(row["predicted_hmri"]) or pd.isna(row["predicted_wbgt"]) or pd.isna(row["priority_score"]):
            continue
        wid = int(row["ward_id"])
        for item in interventions:
            # A completed action type is not repeated for that ward in later runs.
            if completed_quantity(history, wid, item["key"]) > 0:
                continue
            mult = greenery_multiplier(row["green_cover_score_1_to_5"]) if item["type"] == "greenery" else 1.0
            if mult <= 0:
                continue
            for unit_number in range(1, int(item["max_units_per_ward"]) + 1):
                # Diminishing marginal benefit for each additional unit of the same action.
                marginal_fraction = item["hmri_reduction_fraction"] / unit_number
                benefit = float(row["predicted_hmri"]) * (float(row["priority_score"]) / 100.0) * marginal_fraction * mult
                cost = int(item["cost_inr"])
                candidates.append({
                    "ward_id": wid, "item": item, "unit_number": unit_number,
                    "cost": cost, "benefit": benefit,
                    "ratio": benefit / cost
                })

    # Candidate unit N can only be selected after lower unit numbers of the same action.
    selected_units = set()
    while candidates:
        eligible = [
            c for c in candidates
            if c["cost"] <= budget_left
            and (c["unit_number"] == 1 or
                 (c["ward_id"], c["item"]["key"], c["unit_number"] - 1) in selected_units)
        ]
        if not eligible:
            break
        best = max(eligible, key=lambda c: (c["ratio"], c["benefit"], -c["cost"]))
        candidates.remove(best)
        budget_left -= best["cost"]
        selected_units.add((best["ward_id"], best["item"]["key"], best["unit_number"]))
        allocations[best["ward_id"]].append({
            "action_key": best["item"]["key"],
            "action_label": best["item"]["label"],
            "quantity": 1,
            "unit": best["item"]["unit"],
            "unit_cost_inr": best["cost"],
            "total_cost_inr": best["cost"],
            "hmri_reduction_fraction": best["item"]["hmri_reduction_fraction"],
            "estimated_benefit": best["benefit"]
        })

    # Consolidate multiple units of the same intervention into one row/action phrase.
    for wid, items in allocations.items():
        combined = {}
        for a in items:
            k = a["action_key"]
            if k not in combined:
                combined[k] = a.copy()
            else:
                combined[k]["quantity"] += 1
                combined[k]["total_cost_inr"] += a["total_cost_inr"]
                combined[k]["estimated_benefit"] += a["estimated_benefit"]
        allocations[wid] = list(combined.values())

    spent = int(budget_inr) - budget_left
    return allocations, spent, budget_left

allocations, allocated_total, unspent_total = allocate_budget(
    optimizer_wards,
    DEFAULT_BUDGET_INR,
    action_history
)

print(f"Default budget: ₹{DEFAULT_BUDGET_INR:,.0f}")
print(f"Allocated: ₹{allocated_total:,.0f}")
print(f"Unspent: ₹{unspent_total:,.0f}")

assert allocated_total <= DEFAULT_BUDGET_INR

Allocated: ₹5,000,000
Unspent: ₹0


## 11.6 Create one CSV row per ward

In [ ]:
def prior_action_notes(history, ward_id, max_items=2):
    if history.empty:
        return []
    h = history[
        history["ward_id"].astype(str).eq(str(int(ward_id)))
        & history["status"].astype(str).str.lower().eq("completed")
    ].copy()
    if h.empty:
        return []
    h["forecast_date"] = h["forecast_date"].astype(str)
    h = h.sort_values("forecast_date")
    notes = []
    for _, r in h.tail(max_items).iterrows():
        qty = int(pd.to_numeric(pd.Series([r["quantity"]]), errors="coerce").fillna(0).iloc[0])
        label = str(r["action_label"])
        # Concise completion note (roughly 4–5 words).
        note = f"{qty} {label} completed"
        notes.append(note)
    return notes


def make_action_field(row, ward_allocations, history):
    wid = int(row["ward_id"])
    pieces = []
    past = prior_action_notes(history, wid)
    if past:
        pieces.append("Done earlier: " + "; ".join(past))

    new_actions = ward_allocations.get(wid, [])
    if new_actions:
        now = []
        for a in new_actions:
            now.append(f'{a["quantity"]} {a["action_label"]}')
        pieces.append("Now: " + "; ".join(now))
    elif pd.isna(row["predicted_hmri"]) or pd.isna(row["predicted_wbgt"]):
        pieces.append("No forecast received; ₹0 allocated")
    else:
        pieces.append("No new action funded; ₹0 allocated")
    return " | ".join(pieces)


output_rows = []
new_history_rows = []

for _, row in optimizer_wards.iterrows():
    wid = int(row["ward_id"])
    actions = allocations.get(wid, [])
    ward_spend = sum(a["total_cost_inr"] for a in actions)

    before = row["predicted_hmri"]
    # Illustrative estimated reduction; capped to avoid implausible additive reductions.
    reduction_fraction = min(
        0.50,
        sum(a["hmri_reduction_fraction"] * a["quantity"] for a in actions)
    )
    after = float(before) * (1 - reduction_fraction) if pd.notna(before) else np.nan

    output_rows.append({
        "forecast_date": FORECAST_DATE,
        "ward_id": wid,
        "ward_name": row["ward_name"],
        "predicted_wbgt": row["predicted_wbgt"],
        "predicted_hmri": before,
        "priority_score": row["priority_score"],
        "green_cover_score_1_to_5": row["green_cover_score_1_to_5"],
        "budget_allocated_inr": int(ward_spend),
        "estimated_hmri_after_actions": round(after, 2) if pd.notna(after) else np.nan,
        "action": make_action_field(row, allocations, action_history)
    })

    # Per requested prototype scenario, today's recommendations become completed history for tomorrow.
    for a in actions:
        new_history_rows.append({
            "forecast_date": FORECAST_DATE,
            "ward_id": str(wid),
            "action_key": a["action_key"],
            "action_label": a["action_label"],
            "quantity": a["quantity"],
            "unit_cost_inr": a["unit_cost_inr"],
            "total_cost_inr": a["total_cost_inr"],
            "status": "completed"
        })

backend_output = pd.DataFrame(output_rows)
if len(backend_output) != 24:
    raise ValueError(f"Backend output must contain 24 ward rows; found {len(backend_output)}.")
if backend_output["ward_id"].nunique() != 24:
    raise ValueError("Backend output must contain 24 unique ward IDs.")
if backend_output["budget_allocated_inr"].sum() > BUDGET_INR:
    raise ValueError("Allocation exceeds budget.")

display(backend_output.sort_values("priority_score", ascending=False, na_position="last"))
print("Output rows:", len(backend_output))
print("Total allocation: ₹{:,.0f}".format(backend_output["budget_allocated_inr"].sum()))


,forecast_date,ward_id,ward_name,predicted_wbgt,predicted_hmri,priority_score,green_cover_score_1_to_5,budget_allocated_inr,estimated_hmri_after_actions,action
4,2026-11-04,5,E,25.51,48.1,43.838333,2,550000,40.64,Now: 1 water kiosks; 1 shaded rest stations; 1...
2,2026-11-04,3,C,25.78,48.4,43.725000,1,1200000,33.64,Now: 1 water kiosks; 1 shaded rest stations; 1...
7,2026-11-04,9,G_North,25.78,48.8,43.131667,3,550000,41.24,Now: 1 water kiosks; 1 shaded rest stations; 1...
1,2026-11-04,2,B,25.52,47.1,43.050000,1,700000,37.92,Now: 1 water kiosks; 1 shaded rest stations; 1...
8,2026-11-04,8,G_South,25.78,48.0,42.235000,3,550000,40.56,Now: 1 water kiosks; 1 shaded rest stations; 1...
5,2026-11-04,7,F_North,25.78,46.0,39.160000,4,350000,41.17,Now: 1 water kiosks; 1 shaded rest stations
14,2026-11-04,16,M_East,25.78,45.6,38.295000,4,350000,40.81,Now: 1 water kiosks; 1 shaded rest stations
0,2026-11-04,1,A,25.78,44.5,38.051667,2,350000,39.83,Now: 1 water kiosks; 1 shaded rest stations
13,2026-11-04,14,L,24.89,40.3,37.930000,1,100000,38.89,Now: 1 water kiosks
6,2026-11-04,6,F_South,25.78,43.9,37.066667,3,100000,42.36,Now: 1 water kiosks


Output rows: 24
Total allocation: ₹5,000,000


## 11.7 Export backend CSV and update action history

In [ ]:
# CSV is the backend's ward-by-ward output. All action history and new actions are in `action`.
backend_output.to_csv(BACKEND_OUTPUT_CSV, index=False)

# Keep history separately so the next daily run can avoid repeating completed action types.
new_history = pd.DataFrame(new_history_rows, columns=HISTORY_COLUMNS)
updated_history = pd.concat([action_history, new_history], ignore_index=True)
if not updated_history.empty:
    updated_history = updated_history.drop_duplicates(
        subset=["forecast_date", "ward_id", "action_key"], keep="last"
    )
updated_history.to_csv(ACTION_HISTORY_FILE, index=False)

json_out = {
    "forecast_date": FORECAST_DATE,
    "budget_inr": int(BUDGET_INR),
    "allocated_inr": int(backend_output["budget_allocated_inr"].sum()),
    "unspent_inr": int(BUDGET_INR - backend_output["budget_allocated_inr"].sum()),
    "wards": backend_output.astype(object).where(pd.notna(backend_output), None).to_dict(orient="records")
}
with open(BACKEND_OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(json_out, f, indent=2, ensure_ascii=False)

print("Saved backend CSV:", BACKEND_OUTPUT_CSV)
print("Saved action history:", ACTION_HISTORY_FILE)
print("Saved JSON companion:", BACKEND_OUTPUT_JSON)

# Uncomment in Colab to download the backend CSV and the state file.
# files.download(BACKEND_OUTPUT_CSV)
# files.download(ACTION_HISTORY_FILE)


Saved backend CSV: mumbai_heatshield_backend_output.csv
Saved action history: mumbai_heatshield_action_history.csv
Saved JSON companion: mumbai_heatshield_backend_output.json


## Important run notes

- Run the existing notebook cells in order, then run the new Step 11 cells.
- The existing final cell downloads the model file; it can stay where it is, but for a clean notebook it is better to move that download cell to the very end or comment it out until needed.
- In a fresh Colab runtime, `ward_ml_base.csv` and `xgb_wbgt_5d.joblib` must exist or be regenerated/downloaded before the forecasting cells run.
- Keep `mumbai_heatshield_action_history.csv` between daily runs. If you switch devices or reset Colab, download and upload this file; otherwise the optimizer cannot know what it recommended previously.
- The action-history file currently marks recommendations as `completed` to match the requested simulation. For real deployment, this should be replaced by a confirmed status from the field team.
- `estimated_hmri_after_actions` is a scenario estimate based on assumed percentages, not a validated prediction of causal intervention impact.
- The budget allocation uses a greedy heuristic; it respects the budget but does not guarantee a mathematically global optimum.


In [ ]:
# TEST MODE ONLY — synthetic forecasts, not real predictions

import pandas as pd
import numpy as np

np.random.seed(42)

test_forecast_date = "2026-11-05"

test_wards = vulnerability["ward_id"].astype(int).tolist()

test_hmri = pd.DataFrame({
    "ward_id": test_wards,
    "ward_name": vulnerability["ward_name"].tolist(),
    "predicted_wbgt": np.round(
        np.random.uniform(29.0, 36.0, len(test_wards)), 1
    ),
    "HMRI": np.round(
        np.random.uniform(45.0, 90.0, len(test_wards)), 1
    ),
    "forecast_date": test_forecast_date
})

# Replace final_hmri temporarily for testing
final_hmri = test_hmri.copy()

print("TEST MODE: Synthetic forecast data created")
print("Number of wards:", len(final_hmri))
display(final_hmri.head())

TEST MODE: Synthetic forecast data created
Number of wards: 24


,ward_id,ward_name,predicted_wbgt,HMRI,forecast_date
0,1,A,31.6,65.5,2026-11-05
1,2,B,35.7,80.3,2026-11-05
2,3,C,34.1,54.0,2026-11-05
3,4,D,33.2,68.1,2026-11-05
4,5,E,30.1,71.7,2026-11-05


In [ ]:

import pandas as pd

result = pd.read_csv("mumbai_heatshield_backend_output.csv")

print("Total ward rows:", len(result))
print("Unique wards:", result["ward_id"].nunique())
print("Total budget allocated: ₹{:,.0f}".format(
    result["budget_allocated_inr"].sum()
))
print("Budget respected:", result["budget_allocated_inr"].sum() <= BUDGET_INR)

display(
    result[[
        "ward_id",
        "ward_name",
        "predicted_wbgt",
        "predicted_hmri",
        "estimated_hmri_after_actions",
        "priority_score",
        "budget_allocated_inr",
        "action"
    ]].sort_values(
        "priority_score",
        ascending=False,
        na_position="last"
    )
)


Total ward rows: 24
Unique wards: 24
Total budget allocated: ₹5,000,000
Budget respected: True


,ward_id,ward_name,predicted_wbgt,predicted_hmri,estimated_hmri_after_actions,priority_score,budget_allocated_inr,action
4,5,E,25.51,48.1,40.64,43.838333,550000,Now: 1 water kiosks; 1 shaded rest stations; 1...
2,3,C,25.78,48.4,33.64,43.725000,1200000,Now: 1 water kiosks; 1 shaded rest stations; 1...
7,9,G_North,25.78,48.8,41.24,43.131667,550000,Now: 1 water kiosks; 1 shaded rest stations; 1...
1,2,B,25.52,47.1,37.92,43.050000,700000,Now: 1 water kiosks; 1 shaded rest stations; 1...
8,8,G_South,25.78,48.0,40.56,42.235000,550000,Now: 1 water kiosks; 1 shaded rest stations; 1...
5,7,F_North,25.78,46.0,41.17,39.160000,350000,Now: 1 water kiosks; 1 shaded rest stations
14,16,M_East,25.78,45.6,40.81,38.295000,350000,Now: 1 water kiosks; 1 shaded rest stations
0,1,A,25.78,44.5,39.83,38.051667,350000,Now: 1 water kiosks; 1 shaded rest stations
13,14,L,24.89,40.3,38.89,37.930000,100000,Now: 1 water kiosks
6,6,F_South,25.78,43.9,42.36,37.066667,100000,Now: 1 water kiosks
